# 20 Newsgroups Experiment: KATM vs LDA

This notebook compares KATM with LDA (Latent Dirichlet Allocation) baseline on the full 20 newsgroups dataset.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from katm import KATM
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

## Load Full 20 Newsgroups Dataset

In [ ]:
# Load full 20 newsgroups (no subset removal to use all categories)
newsgroups = fetch_20newsgroups(subset='train')

# Use a subset for reasonable runtime (full dataset is large)
documents = newsgroups.data[:1000]
print(f"Loaded {len(documents)} documents")
print(f"Number of categories: {len(newsgroups.target_names)}")
print(f"Categories: {newsgroups.target_names[:5]}... (and {len(newsgroups.target_names)-5} more)")

## LDA Baseline

Train LDA using sklearn's CountVectorizer + LatentDirichletAllocation.

In [ ]:
# Create document-term matrix
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")
print(f"Document-term matrix shape: {doc_term_matrix.shape}")

In [ ]:
# Fit LDA model
n_topics = 10

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online'
)

lda.fit(doc_term_matrix)
print("LDA model fitted!")

In [ ]:
# Print LDA topics
print(f"\n{'='*60}")
print("LDA Topics")
print(f"{'='*60}")

for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-15:-1]
    top_words = [vocab[i] for i in top_words_idx]
    print(f"\nTopic {topic_idx}: {', '.join(top_words)}")

## KATM

Fit KATM on the same data.

In [ ]:
model = KATM(
    kp_algorithm='keybert',
    n_topics=10,
    n_keyphrases=10,
    min_df=3
)

model.fit(documents)
print("KATM model fitted!")

In [ ]:
model.print_topics(n_words=10)

## Qualitative Comparison

Side-by-side comparison of topic word lists.

In [ ]:
print(f"\n{'='*70}")
print(f"{'LDA':^35} | {'KATM':^35}")
print(f"{'='*70}")

# Get LDA topics
lda_topics = []
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-11:-1]
    top_words = [vocab[i] for i in top_words_idx]
    lda_topics.append(top_words)

# Get KATM topics
katm_topics = []
for topic_id in sorted(model.topics_.keys()):
    words = [w for w, _ in model.topics_[topic_id][:10]]
    katm_topics.append(words)

# Print side-by-side
for i in range(min(10, n_topics)):
    lda_str = ', '.join(lda_topics[i][:7]) if i < len(lda_topics) else ''
    katm_str = ', '.join(katm_topics[i][:7]) if i < len(katm_topics) else ''
    print(f"Topic {i}: {lda_str:<35} | {katm_str}")

## Key Differences

**LDA**:
- Uses word co-occurrence statistics only
- Topics tend to be more generic and overlapping
- No semantic understanding of word meaning

**KATM**:
- Uses semantic embeddings via sentence-transformers
- Keyphrase anchors provide better topic coherence
- Topics tend to be more distinct and interpretable
- Can capture semantic similarity between words